In [37]:
import fastf1
import pandas as pd 
import os


if not os.path.exists('../cache'):
    os.makedirs('../cache')

fastf1.Cache.enable_cache('../cache')

print("Laboratorio configurado e cache do FastF1 ativado com sucesso!")

Laboratorio configurado e cache do FastF1 ativado com sucesso!


In [38]:
session = fastf1.get_session(2024, 'São Paulo Grand Prix', 'R') #Puxa os dados da sessao de corrida escolhida dentro do parenteses
session.load()  # Carrega os dados da sessão

core           INFO 	Loading data for São Paulo Grand Prix - Race [v3.8.3]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for lap_count
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...
core        WARNING 	Failed to perform lap accuracy check - all laps marked as inaccurate (driver 23)
core        WARNING 	Failed to perform lap accuracy check - all laps marked as inaccurate (driver 18)
req            INFO 	Using cached data for car_data
req            INFO 	Using cached data for position_data
req            INFO 	Using cached data for weather_data
req            INFO 	Using cached data for race_control_messages
core           INFO 	Finished lo

In [39]:
voltas = session.laps 
voltas.columns

Index(['Time', 'Driver', 'DriverNumber', 'LapTime', 'LapNumber', 'Stint',
       'PitOutTime', 'PitInTime', 'Sector1Time', 'Sector2Time', 'Sector3Time',
       'Sector1SessionTime', 'Sector2SessionTime', 'Sector3SessionTime',
       'SpeedI1', 'SpeedI2', 'SpeedFL', 'SpeedST', 'IsPersonalBest',
       'Compound', 'TyreLife', 'FreshTyre', 'Team', 'LapStartTime',
       'LapStartDate', 'TrackStatus', 'Position', 'Deleted', 'DeletedReason',
       'FastF1Generated', 'IsAccurate'],
      dtype='object')

In [40]:
voltas_vers = voltas.pick_drivers('VER') #Filtra apenas as voltas do piloto escolhido dentro do parenteses
voltas_vers_limpas = voltas_vers.pick_accurate() #Filtra apenas as voltas limpas do piloto escolhido dentro do parenteses
dados_modeloML = voltas_vers_limpas[['LapNumber', 'Stint', 'Compound', 'TyreLife', 'LapTime']].copy()
dados_modeloML['LapTime'] = dados_modeloML['LapTime'].dt.total_seconds() 
dados_modeloML = pd.get_dummies(dados_modeloML, columns=['Compound'], dtype=int) #transforma em binario para o modelo de ML conseguir ler
dados_modeloML.head(5)

,LapNumber,Stint,TyreLife,LapTime,Compound_INTERMEDIATE
1,2.0,1.0,2.0,87.134,1
2,3.0,1.0,3.0,86.240,1
3,4.0,1.0,4.0,86.702,1
4,5.0,1.0,5.0,85.394,1
5,6.0,1.0,6.0,84.311,1


In [41]:
dados_modeloML = dados_modeloML[dados_modeloML['LapTime'] < 90] #Filtra apenas as voltas com tempo menor que 90 segundos, para evitar outliers